<a href="https://colab.research.google.com/github/kwanda2426/projects/blob/main/social_media/Final_tweet_Analytics_DS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Kwanda Mazibuko** - stdnr: 1077167

https://www.youtube.com/watch?v=iJ5bJ_pkFLY

# **Importing Libraries**

In [1]:
!pip install -q tweepy gensim python-louvain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 48.0 MB/s eta 0:00:00


In [2]:
# Importing Libraries - make sure the packages are installed
import os
import tweepy as tw
import re
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from io import BytesIO

import networkx as nx
import community as community_louvain
from community.community_louvain import best_partition

import warnings
warnings.filterwarnings("ignore")

#making sure that we can see all rows and cols
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# **Loading Data**

In [3]:
# Reading data
%%time
FILE_ID = "1lhoPhOLB4fm-IgXlKhZFfC0UiFfHSklZ"
xlsx_url = f"https://docs.google.com/spreadsheets/d/{FILE_ID}/export?format=xlsx"
r = requests.get(xlsx_url)
df_1 = pd.read_excel(BytesIO(r.content), engine = "openpyxl")


CPU times: user 1min 45s, sys: 634 ms, total: 1min 46s
Wall time: 1min 50s


In [4]:
df_1.head(1)

,Query Id,Query Name,Date,Title,Url,Domain,Sentiment,Page Type,Language,Country Code,Continent Code,Continent,Country,City Code,Account Type,Added,Assignment,Author,Category Details,Checked,City,Display URLs,Entity Info,Expanded URLs,Facebook Author ID,Facebook Comments,Facebook Likes,Facebook Role,Facebook Shares,Facebook Subtype,Full Name,Full Text,Gender,Impressions,Instagram Comments,Instagram Followers,Instagram Following,Instagram Interactions Count,Instagram Posts,Interest,Last Assignment Date,Latitude,Location Name,Longitude,Media Filter,Media URLs,Mentioned Authors,Original Url,Priority,Professions,Resource Id,Short URLs,Starred,Station Name,Viewership,Status,Subtype,Thread Author,Thread Created Date,Thread Entry Type,Thread Id,Thread URL,Total Monthly Visitors,X Author ID,X Channel Role,X Followers,X Following,X Replies,X Reply to,X Repost of,X Reposts,X Likes,X Posts,X Verified,Updated,Reach (new),Publication Name,Licenses,Redacted,Redacted Fields,Redaction Reason,Asset Content Id,Asset Thumb Id,Author Verified Type,Avatar,Batch Id,Blog Name,Broadcast Media Url,Is Syndicated,Air Type,Broadcast Type,Media Type,Ad Value,Circulation,Region,Region Code,Daily Visitors,Engagement Type,Hashtags,Item Review,Kicker,Linkedin Comments,Linkedin Engagement,Linkedin Impressions,Linkedin Likes,Linkedin Shares,Linkedin Sponsored,Linkedin Video Views,Parent Post Id,Parent Blog Name,Pub Type,Publisher Sub Type,Rating,Reddit Score,Reddit Score Upvote Ratio,Reddit Comments,Reddit Author Karma,Root Post Id,Root Blog Name,Subreddit,Subreddit Subscribers,Subscriptions,Sub Title,React Score Overall,React Score Emotionality,React Score Harmful,Engagement Score,Subreddit NSFW,Reddit Post Flair,Reddit Author Flair,Subreddit Topics,Reddit Spoiler,Publication Id,Page Type Name,Content Source,Content Source Name,Custom,Bluesky Author Id,Bluesky Followers,Bluesky Following,Bluesky Likes,Bluesky Posts,Bluesky Quotes,Bluesky Replies,Bluesky Reposts,Can Edit Markup,Can Edit Metadata,Can Edit Segmentation,Can Edit Workflow,Copyright,Factiva Attribute Code,Has Full Text,Impact,Instagram Likes,Mention Id,Podcast Audience Estimate,Podcast Duration Ms,Raw Metadata,Reportable,Threads Likes,Threads Quotes,Threads Replies,Threads Reposts,Threads Shares,Threads Views,Tiktok Comments,Tiktok Connected Account,Tiktok Likes,Tiktok Reach,Tiktok Shares,Tiktok Views,Weblog Title,Youtube Comments,Youtube Duration Milliseconds,Youtube Favourites,Youtube Likes,Youtube Subscriber Count,Youtube Video Count,Emotion
0,2003594270,Kenya protests 2025,2025-08-31 21:59:50.0,RT @_James041 Aden Duale is as guilty as F.\n\nGuy has tried ethical card and it has failed.\n\nSHA theft has affected everyone and no one wants to be associated with a thief.\n\nHe has tried hiring affordable bloggers and they have all been humbled by the truth.\n\nThe more a crocodile smiles the more his anus widens.\n\nThere's no escape route for the wicked!\n\n#DualeMustGo #RutoMustGo #DrainTheSwamp,http://twitter.com/kelvinngari62/statuses/1962274155270635732,twitter.com,negative,twitter,en,KEN,AFRICA,Africa,Kenya,KEN.Coast.Mombasa,individual,2025-09-02T09:16:47.214+0000,NaN,kelvinngari62,NaN,False,Mombasa,NaN,"{entityId=13414952, entityConfidence=HIGH, url=https://www.wikidata.org/wiki/Q13414952}, {entityId=43169, entityConfidence=MEDIUM, url=https://www.wikidata.org/wiki/Q43169}, {entityId=2727213, entityConfidence=MEDIUM, url=https://www.wikidata.org/wiki/Q2727213}, {entityId=4682154, entityConfidence=LOW, url=https://www.wikidata.org/wiki/Q4682154}, {entityId=497, entityConfidence=LOW, url=https://www.wikidata.org/wiki/Q497}",NaN,NaN,0,0,NaN,0,NaN,kelvinngari62 (knn),RT @_James041 Aden Duale is as guilty as F.\n\nGuy has tried ethical card and it has failed.\n\nSHA theft has affected everyone and no one wants to be associated with a thief.\n\nHe has tried hiring affordable bloggers and they have all been humbled by the truth.\n\nThe more a crocodile smiles the more his anus widens.\n\nThere's no e

In [ ]:
df_1.shape

(49823, 179)

#### **Pre-Processing**

In [6]:
# Changing column names
df_1.columns = df_1.columns.str.replace(' ', '_', regex=False).str.lower()

In [8]:
df_1['date'] = pd.to_datetime(df_1['date'])
df_june_2025 = df_1[(df_1['date'].dt.month == 6) & (df_1['date'].dt.year == 2025)]

In [11]:
# Extract authors
df_june_2025['tweet_author'] = df_june_2025['author']

# Function to extract mentions from full_text
def extract_mentions(text):
    mentions = re.findall(r'@(\w+)', str(text))
    return list(set(mentions)) # Return unique mentions

# Apply the function to create a new 'mentions' column
df_june_2025['mentions'] = df_june_2025['full_text'].apply(extract_mentions)

# Function to extract retweeted author from full_text
def extract_retweet_author(text):
    retweet_match = re.match(r'RT @(\w+)', str(text)) # Changed regex to look for a space or end of string after username
    if retweet_match:
        return retweet_match.group(1)
    return None

# Apply the function to create a new 'retweeted_author' column
df_june_2025['retweeted_author'] = df_june_2025['full_text'].apply(extract_retweet_author)

Extracted authors, mentions, and retweeted authors.


,tweet_author,mentions,retweeted_author,full_text
17026,Kapher_Oww,[MugureNjehia],MugureNjehia,"RT @MugureNjehia Amkeni....\n\nThey distracted us, passed the finance bill 2025.\nNow they are distracting us with their narrative of violence, anarchy, ethnic profiling so that we can forget that the police killed over 16 people on June 25th this year. \n\nVOTE OUT THIS REGIME ‼️ https://t.co/x33sqezV1F"
17027,njoroge86,[QuincyWandera],QuincyWandera,RT @QuincyWandera Former Attorney General’s son was abducted and DCI and NIS denied that he was taken. It had to take a call to William for them to admit they have him and for William to order 🙄 his release. Unfortunately Ndiangui’s parents don’t have William’s phone number. #RutoMustGo https://t.co/wZlapGu1bc
17028,FamacoFundi,"[Thuso1Africa, WilliamsRuto]",Thuso1Africa,RT @Thuso1Africa Image of a youth in Kenya defying bullets to say enough of the corrupt government destroying their future. Africa must support the youth of Kenya. Williams Ruto and his corrupt government must go. #RutoMustGo @WilliamsRuto https://t.co/6kzTftppG4
17029,CazorlaRoba,[HonOscarSudi],HonOscarSudi,"RT @HonOscarSudi I've seen fiery debates on attacking police, but let's choose peace. Storming stations and seizing arms are unlawful. Gen Z protests were hijacked by gangs fueling chaos. https://t.co/fm8y8kwTqS"
17030,Kapher_Oww,[bevalynekwambo3],bevalynekwambo3,RT @bevalynekwambo3 Happy New Month \n1. Arrest Lagat\n2. Impeach/Fire Murkomen \n3. Disband Ipoa\n4. Fire DCI Amin \n5. Justice for all extra judicial killings \n6. Ruto must go.
...,...,...,...,...
17121,marteen00,[eastersins],eastersins,RT @eastersins Good Morning. Another day for us to Keep in Mind that Eliud Lagat is Still a Free Man and the Govt is Playing Sheep expecting Kenyans to Forget #ArrestEliudLagat #RutoMustGo https://t.co/xpM9SGjFLc
17122,KTNNewsKE,[AshleyMazuri],None,"Families of two young men killed during the June 25th, 2025 Gen Z protests are demanding justice after a postmortem confirmed both died from gunshot wounds.\n@AshleyMazuri\n#KTNPrime https://t.co/1PxWts6hTc"
17123,plitonde,[NdungiGithuku],NdungiGithuku,"RT @NdungiGithuku Our Comrades are innocent.\nThat's why they're not hiding their faces, unlike killer cops and goons when they appear before court.\nThe real goon -head is in Shitty Hole\nWatching innocent men carry his goon-cross in Calvary.\nIpo Siku\n#FreeMuthaiga3\n#FreeNdianguiKinyagi\n#RutoMustGo https://t.co/kr0Wi2UjPI"
17124,njorogemuigai,[NationAfrica],NationAfrica,RT @NationAfrica 'My 26 hours in hell': Mombasa TikToker recounts abduction ahead of June 25 Gen Z protests https://t.co/EeA1QlmjMv https://t.co/tX7lRkqadA


# **Question One**





# **Question Two**

# Task
Analyze the user-mention network from the provided dataset, focusing on tweets from June 2025. This involves preparing the network data by extracting unique authors and mentioned users as nodes, and creating edges for each mention. Then, build a directed graph using `networkx`, compute and analyze network metrics such as degree centrality, betweenness centrality, and clustering coefficient, and display the top users for each metric. Finally, visualize a subset of the network focusing on these top users with a legend, and summarize the insights gained from the analysis and visualization.

## Prepare Network Data

### Subtask:
Extract all unique authors and mentioned users to form the nodes of the network. Then, create a list of edges representing each time an author mentions another user.


**Reasoning**:
To identify all unique participants (nodes) in the network, I will extract unique authors from the 'tweet_author' column and unique mentioned users from the 'mentions' column, then combine these two sets.



In [16]:
unique_authors = df_june_2025['tweet_author'].dropna().unique()
all_mentions = df_june_2025['mentions'].explode().dropna().unique()
nodes = pd.Series(list(set(unique_authors) | set(all_mentions)))

edges = []
for index, row in df_june_2025.iterrows():
    author = row['tweet_author']
    mentions_list = row['mentions']
    if isinstance(mentions_list, list):
        for mentioned_user in mentions_list:
            if pd.notna(author) and pd.notna(mentioned_user):
                edges.append((author, mentioned_user))

print(f"Number of unique nodes: {len(nodes)}")
print(f"Number of edges: {len(edges)}")


Number of unique nodes: 14452
Number of edges: 34732
